In [31]:
file_df = pd.read_csv("emd_b2_phi.csv")
filtered_df = file_df.loc[file_df['TABLE_NAME'] == 'br_ServiceDetail']
filtered_df.to_csv("filtered_df.csv", index=False)

In [15]:
import requests
import pandas as pd

url = "http://localhost:8000/upload_config_from_csv/1/"

input_csv = "filtered_df.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

response = requests.post(url, files=files)

print(response.status_code, response.json())

200 {'success': True, 'message': 'Configuration uploaded successfully.'}


In [253]:
input_csv = "emd_b1_phi.csv"
df = pd.read_csv(input_csv)
tables_b2 = df['TABLE_NAME'].unique()
print(tables_b2)
print(len(tables_b2))


['Address' 'br_VisitDiagnosis' 'br_VisitDiagnosisHistory' 'Claim'
 'Codingsystem' 'DataSourceFacility' 'fctPatientHistory' 'Gender'
 'GenderCodingSystem' 'GenderIdentity' 'GenderIdentityCodingSystem' 'Lab'
 'LabNote' 'LabNoteDetail' 'LabOrder' 'LabPanel' 'LabPatient'
 'LabPatientLabNote' 'LabTestResult' 'LabTestResultFlag'
 'LabTestResultLabNote' 'OBR' 'OBROBX' 'OBX' 'OBXChangeReason' 'OBXManual'
 'OBXManual_Backup_D53959' 'OBXManualHistory'
 'OBXManualHistory_Backup_D53959' 'PatHistCat' 'PatHistCatMaster'
 'PatHistCatPatHistItem' 'PatHistCatPatHistItemMaster' 'PatHistItem'
 'PatHistItemMaster' 'PatHistMedical' 'PatHistMedicalHistory'
 'PatHistMedicalModifier' 'PatHistModifier' 'PatHistSection'
 'PatHistSectionAccessHistory' 'Patient' 'Person' 'PersonEmployers'
 'PersonPhone' 'PersonRace' 'ProcedureMasterInfo' 'Race'
 'RaceCodingSystem' 'RaceSub' 'Visit' 'VisitHistory' 'VisitTypes']
53


In [76]:
join_condition_tables = ['ClinicalPatMedsFillStatus',
'OBXNote',
'OrdReqDiagnosisl',
'OrdReqNote',
'OrdReqOrder',
'PatHistMedicalModifierHistory']

In [77]:
[item for item in  tables_b2 if item in join_condition_tables]

['ClinicalPatMedsFillStatus', 'OBXNote', 'OrdReqNote']

In [5]:
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

20

In [4]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

45

In [5]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

2

In [16]:
tables = ['ClinicalPatMedsFillStatus', 'OBXNote', 'LabNoteDetail', 'OrdReqNote', 'OrdReqDiagnosis']
tables = ['OrdReqDiagnosis']
tables = ['PersonPhone', 'Person', 'LabPatient', 'ProcedureMasterInfo', 'PersonEmployers']
tables = ['PersonEmployers', 'Claim', 'fctPatientHistory']
tables = ['br_VisitDiagnosisHistory']
tables = ['LabNoteDetail']
tables = ['OrdReq']
tables = ['PatMedRecon']
tables = ['br_ServiceDetail']

In [223]:
tables_b3 = ['Notes',
'PatMedRecon',
'OrdReqOrderOrdReqDiagnosis',
'OrdReqOrder',
'InsurancePlan',
'InsuranceProviders',
'NoteType',
'InsurancePlanClass',
'Insurance']
tables = tables_b3

In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/neurodiscoveryai/Documents/deidentification_df/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel, IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process_df.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task, _get_default_table_details_for_ui

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [40]:
TableDetailsModel.objects.filter(table_status=0).count()

8

In [41]:
TableDetailsModel.objects.filter(table_status=1).count()

8

In [42]:
TableDetailsModel.objects.filter(table_status=2).count()

37

In [43]:
TableDetailsModel.objects.filter(table_status=3).count()

0

In [45]:
Task.objects.filter(status=0).count()

10111

In [46]:
Task.objects.filter(status=1).count()

0

In [47]:
Task.objects.filter(status=2).count()

1910

In [39]:
Task.objects.filter(status=-1).count()

0

In [3]:
# Chain.objects.all().delete()

In [4]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION, settings.DEFAULT_OFFSET_VALUE

settings.BATCH_SIZE_DURING_DE_IDENTIFICATION = 100000

In [5]:
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION

100000

In [20]:
# print(no_notes_table)

In [6]:
tables = ['PatHistSection', 'OBXManual_Backup_D53959', 'RaceSub', 'LabPatientLabNote', 'LabTestResultFlag', 
          'OBXManualHistory_Backup_D53959', 'OBROBX', 'Lab', 'PatHistItemMaster', 'LabPanel', 'ProcedureMasterInfo', 
          'Race', 'PersonRace', 'OBX', 'RaceCodingSystem', 'DataSourceFacility', 'PatHistCatPatHistItemMaster', 
          'PatHistItem', 'PatHistSectionAccessHistory', 'Address', 'GenderCodingSystem', 'GenderIdentityCodingSystem', 
          'LabTestResult', 'LabTestResultLabNote', 'Codingsystem', 'OBR', 'PatHistModifier', 'PatHistCat', 
          'OBXChangeReason', 'PatHistCatPatHistItem', 'OBXManual', 'PatHistMedicalModifier', 'PersonEmployers', 
          'LabOrder', 'LabNote', 'VisitTypes', 'Person', 'br_VisitDiagnosisHistory', 'OBXManualHistory', 'Gender', 
          'LabPatient', 'PersonPhone', 'br_VisitDiagnosis', 'PatHistCatMaster', 'GenderIdentity']
#tables = tables_b2
tables = ['PatMedRecon']
completed = []
failed_tables = []
len(tables), len(completed), len(failed_tables)
tables = ['ELIGIBILITYTRACK']
tables = ['PATIENTINSURANCEAUDIT']
tables = ['QMRESULT']
tables = ['INSURANCEPACKAGE']

In [7]:
totalrun = 0
for table in tables:
    print(table)
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, True)
    totalrun += 1
    print(tableobj.rows_count)

2025-07-14 12:31:56,794 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-07-14 12:31:56,829 - deIdentification.nd_logger - INFO - Table INSURANCEPACKAGE dropped from the database.
2025-07-14 12:31:56,831 - deIdentification.nd_logger - INFO - Dropping ignore rows for INSURANCEPACKAGE, postgres


INSURANCEPACKAGE
[]
[]
1903


In [5]:
from django.db.models import Count

grouped_tasks = Task.all_objects.filter(status=0).values('arguments__table_id').annotate(count=Count('id'))
#print(list(counts))

table_ids = [entry['arguments__table_id'] for entry in grouped_tasks if entry['arguments__table_id'] is not None]

table_info = TableDetailsModel.objects.filter(id__in=table_ids).values('id', 'table_name')
table_map = {entry['id']: entry['table_name'] for entry in table_info}

#print(table_info)
#print(table_map)

result = []
for entry in grouped_tasks:
    table_id = entry['arguments__table_id']
    count = entry['count']
    table_name = table_map.get(table_id, 'Unknown')
    result.append({'table_id': table_id, 'table_name': table_name, 'count': count})


total_tasks=0
total_tables = [] 
for r in result:
    total_tasks += r.get('count', 0)
    total_tables.append(r.get('table_name'))
    print("Tasks Left: ", r)
print("Total Tasks:", total_tasks)
print("Total Tables:", len(total_tables))


Tasks Left:  {'table_id': 95, 'table_name': 'CLIENT', 'count': 1}
Tasks Left:  {'table_id': 303, 'table_name': 'INSURANCEPACKAGE', 'count': 2}
Tasks Left:  {'table_id': 227, 'table_name': 'DOCUMENTDIAGNOSIS', 'count': 1}
Total Tasks: 4
Total Tables: 3


In [7]:
Task.objects.filter(status=-1, arguments__table_id=14).last().remarks

AttributeError: 'NoneType' object has no attribute 'remarks'

In [9]:
db, _ = DbDetailsModel.objects.get_or_create(db_name="postgres")
t = TableDetailsModel.objects.get(table_name = 'ELIGIBILITYTRACK', db=db)
config = t.table_details_for_ui
config



{'batch_size': 1000,
 'ignore_rows': {},
 'columns_details': [{'is_phi': False,
   'mask_value': '',
   'column_name': 'CONTEXTID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': 'FACILITYNAME',
   'column_name': 'CONTEXTNAME',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'MASK',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'CONTEXTPARENTCONTEXTID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'ELIGIBILITYTRACKID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'i

In [7]:
config['columns_details'][3]
config['columns_details'][3]['is_phi'] = True
config['columns_details'][3]['de_identification_rule'] = 'ENCOUNTER_ID'
config['columns_details'][3]

{'is_phi': True,
 'mask_value': '',
 'column_name': 'VisitID',
 'ignore_rows': {},
 'add_to_phi_table': False,
 'reference_mapping': {},
 'de_identification_rule': 'ENCOUNTER_ID',
 'column_name_for_phi_table': None}

In [246]:
a = {'conditions': [{'column_name': 'DocumentID',
    'source_column': 'DocumentID',
    'reference_table': 'ClinicalDocuments'}],
  'source_table': 'OrdReqOrder',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

config['reference_mapping'] = a



In [199]:
a = {'conditions': [{'column_name': 'OrdReqID',
    'source_column': 'OrdReqID',
    'reference_table': 'OrdReqClinicalDocuments'},
                   {'column_name': 'DocumentID',
    'source_column': 'DocumentID',
    'reference_table': 'ClinicalDocuments'}],
  'source_table': 'OrdReqDiagnosis',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

config['reference_mapping'] = a



In [174]:
a = {'conditions': [{'column_name': 'OrdReqID',
    'source_column': 'OrdReqID',
    'reference_table': 'OrdReqClinicalDocuments'},
                   {'column_name': 'DocumentID',
    'source_column': 'DocumentID',
    'reference_table': 'ClinicalDocuments'}],
  'source_table': 'OrdReqNote',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

config['reference_mapping'] = a

In [ ]:
a = {'conditions': [{'column_name': 'OBXId',
    'source_column': 'OBXId',
    'reference_table': 'OBX'}],
  'source_table': 'OBXNote',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

config['reference_mapping'] = a

In [ ]:
a = {'conditions': [{'column_name': 'ClinicalPatMedID',
    'source_column': 'ClinicalPatMedID',
    'reference_table': 'ClinicalPatMeds'}],
  'source_table': 'ClinicalPatMedsFillStatus',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

config['reference_mapping'] = a

In [8]:
t.table_details_for_ui = config

In [9]:
t.save()

In [10]:
config

{'batch_size': 1000,
 'ignore_rows': {},
 'columns_details': [{'is_phi': True,
   'mask_value': '',
   'column_name': 'PatientID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'PATIENT_ID',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': '',
   'column_name': 'MedReconDate',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'DATE_OFFSET',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': '',
   'column_name': 'TransitionOfCareDate',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'DATE_OFFSET',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': '',
   'column_name': 'VisitID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'ENCOUNTER_ID',
   'column_name_for_phi_table':

In [11]:
for item in config['columns_details']:
    if item['is_phi']:
        try:
            print(item['column_name'], item['de_identification_rule'],item['mask_value'])
        except Exception as e:
            print(e)

PatientID PATIENT_ID 
VisitID ENCOUNTER_ID 
PostingDate DATE_OFFSET 
Charge MASK Charge
FromDate DATE_OFFSET 
ToDate DATE_OFFSET 
DeductibleAmount MASK DeductibleAmount
HospAdmitDate DATE_OFFSET 
HospDischDate DATE_OFFSET 
DisabilityFromDate DATE_OFFSET 
DisabilityToDate DATE_OFFSET 
SymptomDate DATE_OFFSET 
SimilarDate DATE_OFFSET 
RevisionDateTime DATE_OFFSET 
CurrDateTime DATE_OFFSET 
StatementDate DATE_OFFSET 
CoPayAmtApplied MASK CoPayAmtApplied
ResponsibilityDate DATE_OFFSET 
OrderDate DATE_OFFSET 
LastSeen DATE_OFFSET 
ProcedureAmount MASK ProcedureAmount
InitialTxDate DATE_OFFSET 
NonCoveredCharge MASK NonCoveredCharge
WorkRVU MASK WorkRVU
MalPracticeRVU MASK MalPracticeRVU
NonFacPERVU MASK NonFacPERVU
FacPERVU MASK FacPERVU
ProcedureMasterAmount MASK ProcedureMasterAmount
ChargeScheduleAmount MASK ProcedureAmount
ResubmitCharge MASK ProcedureAmount


In [26]:
IgnoreRowsDeIdentificaiton.objects.filter(table_name='PATIENTINSURANCEAUDIT').count()#.first().row

0

In [189]:
all_left = ['br_ServiceDetailHistory',
'Notes',
'PatMedRecon',
'OrdReqOrderOrdReqDiagnosis',
'Visit',
'PatHistMedicalHistory',
'VisitHistory',
'PatHistMedical',
'OrdReqDiagnosis',
'PatHistSocialHistory',
'LabNoteDetail',
'OrdReqOrder',
'PatHistSocial',
'Patient',
'PatHistSurgicalHistory',
'PatHistSurgical',
'OrdReqClinicalDocuments',
'OrdReq',
'MedicationRxNormFact',
'OrdReqNote',
'InsurancePlan',
'OBXNote',
'InsuranceProviders',
'TypeOfService',
'ClinicalPatMedsFillStatus',
'NoteType',
'InsurancePlanClass',
'Insurance',
'OBROBX_Delete_Backup',
'OrdReqDiagnosisStatus',
'SBVisitNote']

In [193]:
[table for table in all_left if table not in tables_b2]

['Notes',
 'PatMedRecon',
 'OrdReqOrderOrdReqDiagnosis',
 'OrdReqOrder',
 'InsurancePlan',
 'InsuranceProviders',
 'NoteType',
 'InsurancePlanClass',
 'Insurance',
 'OBROBX_Delete_Backup']

In [7]:
db, _ = DbDetailsModel.objects.get_or_create(id=1)
db.db_name

'postgres'